In [1]:
import os
from pathlib import Path

In [2]:
if Path.cwd().name == 'notebooks':
    os.chdir('..')
Path.cwd()

WindowsPath('c:/dev/predictive-maintenance-mlops')

In [3]:
from turbofan.data import RAW_DIR, load_cmapss, split_by_engine

In [4]:
fd1 = load_cmapss(RAW_DIR / 'train_FD001.txt')
fd2 = load_cmapss(RAW_DIR / 'train_FD002.txt')  

tr, val = split_by_engine(fd1)
print(tr.shape[0], val.shape[0])
print(fd2.shape[0])

16561 4070
53759


In [5]:
print("Grouped by unit, sensor 2 mean, tr1", tr['sensor_2'].mean())
print("Grouped by unit, sensor 2 std, tr1", tr['sensor_2'].std())

print("Grouped by unit, sensor 2 mean, val1", val['sensor_2'].mean())
print("Grouped by unit, sensor 2 std, val1", val['sensor_2'].std())

print("Grouped by unit, sensor 2 mean, fd2", fd2['sensor_2'].mean())
print("Grouped by unit, sensor 2 std, fd2", fd2['sensor_2'].std())

Grouped by unit, sensor 2 mean, tr1 642.6776058209045
Grouped by unit, sensor 2 std, tr1 0.49812430609598124
Grouped by unit, sensor 2 mean, val1 642.6944742014741
Grouped by unit, sensor 2 std, val1 0.5076637115632076
Grouped by unit, sensor 2 mean, fd2 579.6723994121915
Grouped by unit, sensor 2 std, fd2 37.289399020462405


In [6]:
import matplotlib.pyplot as plt

for name, df in [("train", tr1), ("val", val1), ("fd2", fd2)]:
    plt.hist(df["sensor_2"], bins=50, alpha=0.5, density=True, label=name)
plt.legend()
plt.show()

NameError: name 'tr1' is not defined

In [ ]:
fd1[["setting_1", "setting_2", "setting_2"]].describe()

,setting_1,setting_2,setting_2
count,20631.000000,20631.000000,20631.000000
mean,-0.000009,0.000002,0.000002
std,0.002187,0.000293,0.000293
min,-0.008700,-0.000600,-0.000600
25%,-0.001500,-0.000200,-0.000200
50%,0.000000,0.000000,0.000000
75%,0.001500,0.000300,0.000300
max,0.008700,0.000600,0.000600


In [ ]:
fd2[["setting_1", "setting_2", "setting_2"]].describe()

,setting_1,setting_2,setting_2
count,53759.000000,53759.000000,53759.000000
mean,23.998407,0.572056,0.572056
std,14.747376,0.310016,0.310016
min,0.000000,0.000000,0.000000
25%,10.004600,0.250700,0.250700
50%,25.001300,0.700000,0.700000
75%,41.998000,0.840000,0.840000
max,42.008000,0.842000,0.842000


In [ ]:
from evidently import Report
from evidently.presets import DataDriftPreset
from turbofan.features import FEATURE_SENSORS

Path("reports").mkdir(exist_ok=True)

report = Report([DataDriftPreset()])

control = report.run(current_data=val[FEATURE_SENSORS], reference_data=tr[FEATURE_SENSORS])
control.save_html("reports/drift_control.html")

drifted = report.run(current_data=fd2[FEATURE_SENSORS], reference_data=fd1[FEATURE_SENSORS])
drifted.save_html("reports/drift_fd002.html")

In [ ]:
# result = drifted.dict()
# print(result.keys)

# first = result["metrics"][0]
# print(first["metric_name"])
# print(first["value"])

controlled_result = control.dict()
print(controlled_result.keys())

controlled_first = controlled_result["metrics"][0]
print(controlled_first["metric_name"])
print(controlled_first["value"])

dict_keys(['metrics', 'tests'])
DriftedColumnsCount(drift_share=0.5)
{'count': 0.0, 'share': 0.0}


In [7]:
from turbofan.drift import run_drift_report, summarize

print(summarize(run_drift_report(tr, val)))
print(summarize(run_drift_report(tr, fd2)))

{'drifted_columns': 0, 'share': 0.0, 'scores': {'sensor_2': 0.045, 'sensor_3': 0.048, 'sensor_4': 0.048, 'sensor_6': 0.009, 'sensor_7': 0.058, 'sensor_8': 0.059, 'sensor_9': 0.075, 'sensor_11': 0.051, 'sensor_12': 0.045, 'sensor_13': 0.059, 'sensor_14': 0.086, 'sensor_15': 0.031, 'sensor_17': 0.029, 'sensor_20': 0.054, 'sensor_21': 0.061}}
{'drifted_columns': 15, 'share': 1.0, 'scores': {'sensor_2': 126.489, 'sensor_3': 28.022, 'sensor_4': 22.691, 'sensor_6': 7338.86, 'sensor_7': 308.479, 'sensor_8': 2253.764, 'sensor_9': 23.84, 'sensor_11': 17.14, 'sensor_12': 347.427, 'sensor_13': 747.747, 'sensor_14': 3.947, 'sensor_15': 23.666, 'sensor_17': 29.035, 'sensor_20': 100.233, 'sensor_21': 100.716}}
